# TRANSFORMACIÓN DE DATOS

## IMPORTAR PAQUETES

In [5]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import OrdinalEncoder
from sklearn.preprocessing import LabelEncoder
from category_encoders import TargetEncoder
from sklearn.preprocessing import KBinsDiscretizer
from sklearn.preprocessing import Binarizer
from sklearn.preprocessing import PowerTransformer
from sklearn.preprocessing import QuantileTransformer
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import RobustScaler
from sklearn.preprocessing import MaxAbsScaler
from imblearn.under_sampling import RandomUnderSampler
from imblearn.over_sampling import RandomOverSampler

#Automcompletar rápido
%config IPCompleter.greedy=True

## IMPORTAR LOS DATOS

1.- Sustituir la ruta del proyecto.

In [6]:
ruta_proyecto = 'C:/Users/Lenovo/Documents/portafolio_ds/automation/LEAD_KAGGLE'

2.- Nombrar los archivos de datos.

In [7]:
nombre_cat = 'cat_resultado_eda.pickle'
nombre_num = 'num_resultado_eda.pickle'

3.- Cargar los datos.

In [8]:
cat = pd.read_pickle(ruta_proyecto + '/02_Datos/03_Trabajo/' + nombre_cat).reset_index(drop=True)
num = pd.read_pickle(ruta_proyecto + '/02_Datos/03_Trabajo/' + nombre_num).reset_index(drop=False)

4.- Separar la target.

In [9]:
target = num[['converted']].copy().reset_index(drop=True)

## TRANSFORMACIÓN DE CATEGÓRICAS

### One Hot Encoding

#### Variables a aplicar OHE

In [10]:
var_ohe = ['lead_origin','lead_source','last_activity','country','specialization','what_is_your_current_occupation','tags','city','last_notable_activity']

#### Instanciar

In [11]:
ohe = OneHotEncoder(sparse_output = False, handle_unknown='ignore')

#### Entrenar y aplicar

In [12]:
cat_ohe = ohe.fit_transform(cat[var_ohe])

#### Guardar como dataframe

In [13]:
cat_ohe = pd.DataFrame(cat_ohe, columns = ohe.get_feature_names_out())

### Ordinal Encoding

#### Variables a aplicar OE

In [14]:
var_oe = ['asymmetrique_activity_index','asymmetrique_profile_index']

#### Orden de los valores de las variables

In [15]:
#Orden de la primera variable
orden_activity = ['03.Low','02.Medium','01.High']

#Orden de la segunda variable
orden_profile = ['Unknown','03.Low','02.Medium','01.High']

#### Instanciar

In [16]:
oe = OrdinalEncoder(categories = [orden_activity,orden_profile],
                    handle_unknown = 'use_encoded_value',
                    unknown_value = 10)

#### Entrenar y aplicar

In [17]:
cat_oe = oe.fit_transform(cat[var_oe])

#### Guardar como dataframe

In [18]:
#Añadir sufijos a los nombres
nombres_oe = [variable + '_oe' for variable in var_oe]

#Guardar como dataframe
cat_oe = pd.DataFrame(cat_oe, columns = nombres_oe)

## UNIFICAR DATASETS TRANSFORMADOS

### Meter en una lista todos los dataframes generados

In [19]:
dataframes = []
dataframes.extend(value for name, value in locals().items() if name.startswith('cat_') or name.startswith('num'))

### Unir todos los dataframes

In [20]:
df = pd.concat(dataframes, axis = 1)

## REESCALAR VARIABLES

### Con Min-Max

#### Variables a reescalar con Min-Max

In [21]:
df.iloc[:,1:8:].columns

Index(['lead_number', 'converted', 'total_visits',
       'total_time_spent_on_website', 'page_views_per_visit',
       'asymmetrique_activity_score', 'asymmetrique_profile_score'],
      dtype='object')

In [22]:
var_mms = ['lead_number','total_visits','total_time_spent_on_website','page_views_per_visit','asymmetrique_activity_score','asymmetrique_profile_score']

#### Instanciar

In [23]:
mms = MinMaxScaler()

#### Entrenar y aplicar

In [24]:
df_mms = mms.fit_transform(df[var_mms])

#### Guardar como dataframe

In [25]:
#Añadir sufijos a los nombres
nombres_mms = [variable + '_mms' for variable in var_mms]

#Guardar como dataframe
df_mms = pd.DataFrame(df_mms,columns = nombres_mms)

## UNIFICAR DATASETS REESCALADOS

### Crear una lista con los dataframes a incluir en el tablón analítico

In [26]:
Prospect_ID = df['Prospect ID'].to_frame()
Prospect_ID

,Prospect ID
0,7927b2df-8bba-4d29-b9a2-b6e0beafe620
1,2a272436-5132-4136-86fa-dcc88c88f482
2,8cc8c611-a219-4f35-ad23-fdfd2656bd8a
3,0cc2df48-7cf4-4e39-9de9-19797f9b38cc
4,3256f628-e534-4826-9d63-4a8b88782852
...,...
5935,3f715465-2546-47cd-afa8-8b8dc63b8b43
5936,c0b25922-511f-4c56-852e-ced210a45447
5937,82a7005b-7196-4d56-95ce-a79f937a158d
5938,5330a7d1-2f2b-4df4-85d6-64ca2f6b95b9


In [70]:
incluir = [Prospect_ID, cat_ohe, cat_oe, df_mms, target]

### Unir todos los dataframes en el tablón analítico

In [71]:
df_tablon = pd.concat(incluir, axis = 1).set_index('Prospect ID')

In [72]:
df_tablon.info()

<class 'pandas.core.frame.DataFrame'>
Index: 5940 entries, 7927b2df-8bba-4d29-b9a2-b6e0beafe620 to 571b5c8e-a5b2-4d57-8574-f2ffb06fdeff
Data columns (total 66 columns):
 #   Column                                                Non-Null Count  Dtype  
---  ------                                                --------------  -----  
 0   lead_origin_API                                       5940 non-null   float64
 1   lead_origin_Landing Page Submission                   5940 non-null   float64
 2   lead_origin_Lead Add Form                             5940 non-null   float64
 3   lead_origin_OTHER                                     5940 non-null   float64
 4   lead_source_Direct Traffic                            5940 non-null   float64
 5   lead_source_Google                                    5940 non-null   float64
 6   lead_source_OTHER                                     5940 non-null   float64
 7   lead_source_Olark Chat                                5940 non-null   float64
 

In [73]:
pd.set_option('display.max_rows', None)

In [74]:
#Revisión general de los datos
df_tablon.describe().T

,count,mean,std,min,25%,50%,75%,max
lead_origin_API,5940.0,0.404040,0.490747,0.0,0.000000,0.000000,1.000000,1.0
lead_origin_Landing Page Submission,5940.0,0.511785,0.499903,0.0,0.000000,1.000000,1.000000,1.0
lead_origin_Lead Add Form,5940.0,0.077441,0.267313,0.0,0.000000,0.000000,0.000000,1.0
lead_origin_OTHER,5940.0,0.006734,0.081791,0.0,0.000000,0.000000,0.000000,1.0
lead_source_Direct Traffic,5940.0,0.259933,0.438634,0.0,0.000000,0.000000,1.000000,1.0
lead_source_Google,5940.0,0.322054,0.467303,0.0,0.000000,0.000000,1.000000,1.0
lead_source_OTHER,5940.0,0.038889,0.193346,0.0,0.000000,0.000000,0.000000,1.0
lead_source_Olark Chat,5940.0,0.199663,0.399781,0.0,0.000000,0.000000,0.000000,1.0
lead_source_Organic Search,5940.0,0.120539,0.325618,0.0,0.000000,0.000000,0.000000,1.0
lead_source_Reference,5940.0,0.058923,0.235500,0.0,0.000000,0.000000,0.000000,1.0


In [75]:
pd.reset_option('display.max_rows')

## GUARDAR DATASET TRAS TRANSFORMACIÓN DE DATOS

En formato pickle para no perder las modificaciones de metadatos.

In [76]:
#Definir los nombres del archivo
ruta_df_tablon = ruta_proyecto + '/02_Datos/03_Trabajo/' + 'df_tablon.pickle'

In [77]:
#Guardar los archivos
df_tablon.to_pickle(ruta_df_tablon)